# C3 — Hồi quy logistic trên dữ liệu khả tách (Kiên)
Huấn luyện có và không có chính quy hóa; vẽ quỹ đạo ‖w_k‖ theo vòng lặp; đối chiếu B.2.
Chỉ dùng dữ liệu tổng hợp. Mã thuật toán: src/logistic.py (IRLS).

In [ ]:
import sys
sys.path.insert(0, "../src")
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.svm import SVC

from logistic import fit_irls

SEEDS = [42, 43, 44, 45, 46]
LAMS = [1e-3, 1e-2, 1e-1]
N, MU, K_NEWTON = 100, 2.0, 40


def make_data(seed, n=N, mu=MU):
    """Hai cụm Gauss N(∓mu·1, I) trong R², mỗi cụm n/2 điểm. Trả về X (n, 2), y (n,) ∈ {0,1}."""
    rng = np.random.default_rng(seed)
    X = np.r_[rng.normal(-mu, 1, (n // 2, 2)), rng.normal(mu, 1, (n // 2, 2))]
    y = np.r_[np.zeros(n // 2), np.ones(n // 2)]
    return X, y


def b2_lower_bound(L, R):
    """Cận dưới ở B.2: ‖θ‖ ≥ (1/R)·log(1/(e^L − 1)), có nghĩa khi L < log 2 (L là loss TỔNG)."""
    L = np.asarray(L, dtype=float)
    out = np.full_like(L, np.nan)
    ok = L < np.log(2)
    out[ok] = np.log(1.0 / np.expm1(L[ok])) / R
    return out

In [ ]:
# Kiểm tra dữ liệu khả tách (SVM tuyến tính gần như cứng đạt 100% trên tập huấn luyện)
for s in SEEDS:
    X, y = make_data(s)
    acc = SVC(kernel="linear", C=1e6).fit(X, y).score(X, y)
    assert acc == 1.0, f"seed {s} không khả tách"
print("5/5 bộ dữ liệu khả tách tuyến tính")

In [ ]:
# IRLS: λ = 0 và λ > 0
rows, traj = [], {}
for s in SEEDS:
    X, y = make_data(s)
    n = len(y)
    R = np.max(np.linalg.norm(np.c_[X, np.ones(n)], axis=1))   # R = max ‖(x_i, 1)‖
    for lam in [0.0] + LAMS:
        w, b, h = fit_irls(X, y, lam=lam, max_iter=K_NEWTON, tol=0.0 if lam == 0 else 1e-10)
        theta_norm = np.array([np.linalg.norm(np.r_[wk, bk]) for wk, bk in zip(h["w"], h["b"])])
        # loss dữ liệu dạng TỔNG như trong B.2 (bỏ phần phạt): L = n·(loss − λ/2‖w‖²)
        L_sum = n * (np.array(h["loss"]) - 0.5 * lam * np.array(h["norm_w"]) ** 2)
        traj[(s, lam)] = dict(theta=theta_norm, L=L_sum, bound=b2_lower_bound(L_sum, R),
                              grad=np.array(h["grad_norm"]))
        # λ > 0 hội tụ sớm: lịch sử ngắn hơn K_NEWTON, giá trị sau hội tụ giữ nguyên
        at = lambda a, k: a[min(k, len(a) - 1)]
        k_conv = next((k for k, g in enumerate(h["grad_norm"]) if g < 1e-10), None)  # λ=0: ‖g‖ nhỏ dần dù θ không hội tụ
        rows.append(dict(seed=s, lam=lam, k_hoi_tu=k_conv,
                         theta_k10=at(theta_norm, 10), theta_k20=at(theta_norm, 20), theta_cuoi=theta_norm[-1],
                         L_k10=at(L_sum, 10), L_k20=at(L_sum, 20), L_cuoi=L_sum[-1],
                         bound_cuoi=traj[(s, lam)]["bound"][-1], R=R,
                         co_nan=bool(np.any(~np.isfinite(theta_norm)))))
irls = pd.DataFrame(rows)

# Độ dốc của ‖θ_k‖ theo k ở nửa sau quỹ đạo (λ = 0): Newton làm ‖θ‖ tăng TUYẾN TÍNH
slopes = [np.polyfit(np.arange(20, K_NEWTON + 1), traj[(s, 0.0)]["theta"][20:], 1)[0] for s in SEEDS]
print("Độ dốc ‖θ_k‖ (λ=0, k=20..40): %.2f ± %.2f mỗi vòng" % (np.mean(slopes), np.std(slopes, ddof=1)))

# Kiểm tra cận B.2 ở mọi vòng có L < log 2
viol = sum(int(np.nansum(t["theta"] < t["bound"] - 1e-9)) for (s, lam), t in traj.items() if lam == 0)
print("Số vòng vi phạm cận B.2 (λ=0, mọi seed):", viol)

summary = irls.groupby("lam").agg(
    k_hoi_tu_mean=("k_hoi_tu", "mean"), k_hoi_tu_std=("k_hoi_tu", "std"),
    theta_k10_mean=("theta_k10", "mean"), theta_k10_std=("theta_k10", "std"),
    theta_k20_mean=("theta_k20", "mean"), theta_k20_std=("theta_k20", "std"),
    theta_cuoi_mean=("theta_cuoi", "mean"), theta_cuoi_std=("theta_cuoi", "std"),
    L_k10_mean=("L_k10", "mean"), L_k10_std=("L_k10", "std"),
    L_k20_mean=("L_k20", "mean"), L_k20_std=("L_k20", "std"),
    L_cuoi_mean=("L_cuoi", "mean"), L_cuoi_std=("L_cuoi", "std"),
    co_nan=("co_nan", "any"),
).reset_index()
print(summary.T)

In [ ]:
# Lưu kết quả
irls.to_csv("../results/C3_irls.csv", index=False)
summary.to_csv("../results/C3_tom_tat.csv", index=False)

In [ ]:
# Hình: quỹ đạo ‖θ_k‖ và loss
BLUE, ORANGE, AQUA, YELLOW, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#222222", "#8a8a8a"
LAM_COL = {1e-3: ORANGE, 1e-2: AQUA, 1e-1: YELLOW}
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": MUTED, "axes.grid": True, "grid.color": "#e6e6e6",
                     "grid.linewidth": 0.6, "lines.linewidth": 2})
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
k = np.arange(K_NEWTON + 1)

# (a) IRLS: ‖θ_k‖
for s in SEEDS:
    t = traj[(s, 0.0)]
    ax[0].plot(k, t["theta"], color=BLUE, lw=2 if s == 42 else 1, alpha=1 if s == 42 else 0.35,
               label="λ = 0 (đậm: seed 42; nhạt: 43–46)" if s == 42 else None)
ax[0].plot(k, traj[(42, 0.0)]["bound"], color=INK, lw=1.2, ls="--", label="cận dưới B.2 (seed 42, λ = 0)")
for lam in LAMS:
    t = traj[(42, lam)]["theta"]
    ax[0].plot(np.arange(len(t)), t, color=LAM_COL[lam], label=f"λ = {lam:g} (hội tụ)")
ax[0].set(title="(a) IRLS: chuẩn $\\|\\theta_k\\|$", xlabel="Vòng lặp $k$", ylabel="$\\|(w_k, b_k)\\|$")
ax[0].legend(frameon=False, loc="upper left", fontsize=8)

# (b) IRLS: loss dữ liệu L(θ_k) (thang log)
ax[1].semilogy(k, traj[(42, 0.0)]["L"], color=BLUE, label="λ = 0")
for lam in LAMS:
    t = traj[(42, lam)]["L"]
    ax[1].semilogy(np.arange(len(t)), t, color=LAM_COL[lam], label=f"λ = {lam:g}")
ax[1].set(title="(b) IRLS: loss dữ liệu $L(\\theta_k)$", xlabel="Vòng lặp $k$", ylabel="$L$ (tổng, thang log)")
ax[1].legend(frameon=False)

for a in ax:
    a.tick_params(colors=INK)
fig.tight_layout()
fig.savefig("../results/C3_norm_w.png", dpi=300)
print("Đã lưu results/C3_norm_w.png")